# Churn Analysis - MongoDB Project (Part B)

<img src="https://media.istockphoto.com/id/1188462768/photo/notepad-with-churn-percent-near-marker-over-desk.jpg?s=612x612&w=0&k=20&c=1cP2pmT6oHcyrAvnMbivVx13jJyw-8Cr2FJos3JblJI=" />

## Team Members
*Idan Arbel - 204344022*  

*Yovel Yefet - 318987815*  

---

## Project Goal - Part B

Pull a fresh batch of customer records from the **MongoDB `Tech.Customers`** collection, clean them so they match the original `churn.csv` schema, run them through the **best model from Part A** (tuned **Random Forest**), and export the result as `churn.csv` with a predicted `Churn` column.

## 1. Imports & MongoDB Connection

In [ ]:
import numpy as np
import pandas as pd
from pandas import json_normalize
import pymongo
from pymongo import MongoClient

# Machine Learning imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
client = MongoClient('mongodb://localhost:27017/')
client.list_database_names()

In [ ]:
db = client['Tech']
db.list_collection_names()

## 2. MQL Query - Documents With All Fields, Excluding `_id`

**Requirements (from the assignment):**
- (a) Return only documents that have **all** of the expected fields.
- (b) Return all fields **except `_id`**.

**Schema observed in MongoDB Compass:**  
Top-level fields -> `customerID, gender, SeniorCitizen, Partner, Dependents, tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, Services`.  
Nested under `Services` -> `PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies`.

We use the MongoDB `$exists: True` operator on every required field (including nested paths) and a projection `{'_id': 0}` to drop the auto-generated ObjectId.

In [ ]:
# (a) Filter - every required field must exist on the document.
required_fields = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'tenure', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges',
    # nested Services sub-document
    'Services',
    'Services.PhoneService', 'Services.MultipleLines', 'Services.InternetService',
    'Services.OnlineSecurity', 'Services.OnlineBackup', 'Services.DeviceProtection',
    'Services.TechSupport', 'Services.StreamingTV', 'Services.StreamingMovies',
]

mql_filter = {field: {'$exists': True} for field in required_fields}

# (b) Projection - exclude only `_id`, keep every other field.
mql_projection = {'_id': 0}

print('Filter :', mql_filter)
print('Project:', mql_projection)

In [ ]:
# Apply the MQL query and count the matched documents.
result = db.Customers.find(mql_filter, mql_projection)
docs = list(result)
print(f'Documents returned by MQL query: {len(docs):,}')
# preview the first two documents
docs[:2]

## 3. Load Collection Into a Pandas DataFrame

Using the MQL query above, we materialise the cursor into a list and convert it into a `pandas.DataFrame`.

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.DataFrame(docs)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

## 4. Clean & Adjust Problematic (Non-Scalar) Columns

MongoDB Compass shows two non-scalar columns that won't work with scikit-learn / `to_csv` out of the box:

| Column | Type in MongoDB | Action |
|--------|-----------------|--------|
| `PaperlessBilling` | **array** (e.g. `['Yes']`) | unwrap the single element -> scalar `'Yes'` / `'No'` |
| `Services`         | **embedded document** | flatten its 9 keys into top-level columns |

After this step every column is a scalar and the DataFrame matches the schema of the original `churn.csv`.

In [ ]:
# 4a) Unwrap the PaperlessBilling array -> scalar value.
df['PaperlessBilling'] = df['PaperlessBilling'].apply(
    lambda v: v[0] if isinstance(v, list) and len(v) > 0 else v
)
df['PaperlessBilling'].unique()

In [ ]:
# 4b) Flatten the embedded Services document into top-level columns.
services_df = json_normalize(df['Services'])
print('New columns extracted from Services:', list(services_df.columns))

df = pd.concat([df.drop(columns=['Services']), services_df], axis=1)
df.head()

In [ ]:
# Reorder columns to match the original churn.csv layout.
csv_column_order = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges'
]
df = df[csv_column_order]
print('Shape after cleaning:', df.shape)
df.head()

In [ ]:
is_scalar = df.map(lambda v: not isinstance(v, (list, dict))).all()
print(is_scalar)


## 5. Snapshot the Raw DataFrame

We keep an untouched copy (`raw_df`) of the cleaned-but-unencoded DataFrame. After running the ML model we will append the predicted `Churn` column to this raw copy and export it - keeping the CSV in the **exact same human-readable format** as the original `churn.csv`.

In [ ]:
raw_df = df.copy()
print('raw_df shape:', raw_df.shape)
raw_df.head()

## 6. Prep the DataFrame for the ML Model (Code From Part A)

We re-use the data-preparation pipeline from **Python Project - Part A**:
1. Convert `TotalCharges` to numeric (blank -> NaN -> fill with `MonthlyCharges`).
2. Drop `gender` (no predictive signal in Part A EDA).
3. Encode binary Yes/No columns to 1/0.
4. Collapse `"No internet service"` / `"No phone service"` -> `"No"`, then encode to 1/0.
5. Engineer `services_count` (sum of 7 binary service columns).
6. Engineer `TenureGroup` (ordinal bins 0-3), then drop `tenure`.
7. One-hot encode `InternetService`, `Contract`, `PaymentMethod`, `TenureGroup`.
8. Cast all feature columns to `float`.

In [ ]:
# 6.1 - TotalCharges: blank strings -> NaN -> fill with MonthlyCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

# 6.2 - Drop non-predictive gender column
df = df.drop(columns=['gender'])

# 6.3 - Binary Yes/No columns
binary_yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_yes_no_cols:
    df[col] = (df[col] == 'Yes').astype(int)

# 6.4 - Multi-category service columns -> binary
multi_category_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in multi_category_cols:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    df[col] = (df[col] == 'Yes').astype(int)

# 6.5 - services_count
service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['services_count'] = df[service_cols].sum(axis=1)

# 6.6 - TenureGroup (ordinal), then drop tenure
df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 36, 72], labels=[0, 1, 2, 3]).astype(int)
df = df.drop(columns=['tenure'])

# 6.7 - One-hot encoding
df = pd.get_dummies(df, columns=['InternetService'])
df = pd.get_dummies(df, columns=['Contract'])
df = pd.get_dummies(df, columns=['PaymentMethod'])
df = pd.get_dummies(df, columns=['TenureGroup'])

# Convert any boolean dummy columns to 0/1 ints
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# 6.8 - Cast feature columns (everything except customerID) to float
cols_to_convert = df.columns.difference(['customerID'])
df[cols_to_convert] = df[cols_to_convert].astype(float)

print('Prepared DataFrame shape:', df.shape)
df.head()

## 7. Use the Best Model From Part A and Predict `Churn`

Part A already ran the full hyper-parameter search and identified the winner:

| Search method | Best parameters | Test accuracy |
|---------------|-----------------|---------------|
| `GridSearchCV` | `max_depth=10, n_estimators=100` | 0.8004 |
| **`RandomizedSearchCV`** | **`max_depth=10, n_estimators=200`** | **0.8011** |

So the **best model** is a **Random Forest with `n_estimators=200`, `max_depth=10`, `random_state=1`** (from Part A - Section 13 / Section 14).

**Two data flows in this section:**
- **Training** — Part A labeled dataset (`churn_labeled.csv`, 7,043 rows, ground-truth `Churn` column). We must read this separately because the MongoDB collection has **no `Churn` labels**.
- **Prediction** — `df` from **Section 6** (30 features, 24,221 rows). This is the unlabeled MongoDB data we want to score.

> **Setup**: Place `churn.csv` (from Part A) next to this notebook. This avoids a naming conflict with the `churn.csv` output written in Section 8.

In [ ]:
# ── 7a. Train the best model on Part A labeled data ───────────────────────
# churn_labeled.csv = the original Part A churn.csv (7,043 rows, has Churn column).
# This is completely separate from df (Section 6) and from the output written in Section 8.
PART_A_LABELED_CSV = 'churn_labeled.csv'
part_a_df = pd.read_csv(PART_A_LABELED_CSV)

# Target encoding:
part_a_df.loc[part_a_df.Churn == 'No',  'Churn'] = '0'
part_a_df.loc[part_a_df.Churn == 'Yes', 'Churn'] = '1'

# Apply the same Part A preprocessing pipeline:
part_a_df['TotalCharges'] = pd.to_numeric(part_a_df['TotalCharges'], errors='coerce')
part_a_df['TotalCharges'] = part_a_df['TotalCharges'].fillna(part_a_df['MonthlyCharges'])
part_a_df = part_a_df.drop(columns=['gender'])

for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    part_a_df[col] = (part_a_df[col] == 'Yes').astype(int)

multi = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
         'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in multi:
    part_a_df[col] = part_a_df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    part_a_df[col] = (part_a_df[col] == 'Yes').astype(int)

part_a_df['services_count'] = part_a_df[multi].sum(axis=1)
part_a_df['TenureGroup'] = pd.cut(part_a_df['tenure'], bins=[-1, 12, 24, 36, 72], labels=[0, 1, 2, 3]).astype(int)

part_a_df = pd.get_dummies(part_a_df, columns=['InternetService'])
part_a_df = pd.get_dummies(part_a_df, columns=['Contract'])
part_a_df = pd.get_dummies(part_a_df, columns=['PaymentMethod'])
part_a_df = pd.get_dummies(part_a_df, columns=['TenureGroup'])

bool_cols = part_a_df.select_dtypes(include='bool').columns
part_a_df[bool_cols] = part_a_df[bool_cols].astype(int)

part_a_df = part_a_df.drop(columns=['tenure'])
cols_to_convert = part_a_df.columns.difference(['customerID'])
part_a_df[cols_to_convert] = part_a_df[cols_to_convert].astype(float)

# Train / test split for accuracy validation:
label, id_col = 'Churn', 'customerID'
test_size = int(len(part_a_df) * 0.2)
train_split, test_split = train_test_split(part_a_df, test_size=test_size,
                                           random_state=0, shuffle=True,
                                           stratify=part_a_df[label])
x_train = train_split.drop([label, id_col], axis=1)
y_train = train_split[label]
x_test  = test_split.drop([label, id_col], axis=1)
y_test  = test_split[label]

# Best model (RandomizedSearchCV winner from Part A):
best_final_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=1)
best_final_model.fit(x_train, y_train)

test_acc = accuracy_score(y_test, best_final_model.predict(x_test))
print(f'Best model from Part A:  RandomForestClassifier(n_estimators=200, max_depth=10)')
print(f'Test accuracy on Part A held-out set: {test_acc:.4f}')

In [ ]:
# ── 7b. Predict on df from Section 6 (30 features, 24,221 MongoDB records) ─
X_predict = df.drop(columns=['customerID'])
X_predict = X_predict.reindex(columns=x_train.columns, fill_value=0)

predictions = best_final_model.predict(X_predict)
df['Churn'] = predictions.astype(int)

print('Prediction distribution:')
print(df['Churn'].value_counts())
df[['customerID', 'Churn']].head(10)

## 8. Export the DataFrame to `churn.csv`

This step is **specific to Part B** - Part A only saved a numeric, encoded `churn_df_cleaned.csv`. The Part B assignment requires the final CSV to be in **the same format as the original `churn.csv`** (Part A's input file): original column order, original string values (`Yes`/`No`, `Month-to-month`, `Fiber optic`, ...) with the predicted `Churn` mapped back to `Yes`/`No`.

That's why we attach the predictions to the `raw_df` snapshot taken in Section 5 (pre-encoding) instead of the numeric `df` used for the model.

In [ ]:
# Attach the predicted Churn column to the un-encoded (raw) DataFrame.
final_df = raw_df.copy()
final_df['Churn'] = pd.Series(predictions).map({0: 'No', 1: 'Yes'}).values

# Re-order columns to match churn.csv exactly.
final_columns = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges', 'Churn'
]
final_df = final_df[final_columns]
final_df.head()

In [ ]:
# Save without the index - same format as the original churn.csv.
final_df.to_csv('churn.csv', index=False)
print(f'Exported {len(final_df):,} rows to churn_partB.csv')
print('Columns:', list(final_df.columns))

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Connected to MongoDB and selected `Tech.Customers`. |
| 2 | Built an MQL query that returns only documents with **all** required fields (including nested `Services.*` paths), projecting everything **except `_id`**. |
| 3 | Loaded the result into a pandas DataFrame. |
| 4 | Cleaned non-scalar columns: unwrapped `PaperlessBilling` (array -> scalar) and flattened `Services` (sub-document -> 9 top-level columns). |
| 5 | Snapshotted the raw, human-readable DataFrame for the final export. |
| 6 | Re-applied the **Part A** data-prep pipeline. |
| 7 | Used the best model from Part A (RandomizedSearchCV winner: `RandomForestClassifier(n_estimators=200, max_depth=10)`); fit it on the labelled `churn.csv` and predicted `Churn` for the MongoDB customers. |
| 8 | Exported the result as `churn.csv` in the exact same format as the original (with the predicted `Churn` column). |